In [82]:
from ucimlrepo import fetch_ucirepo 
  
# fetch dataset 
spambase = fetch_ucirepo(id=94) 
  
# data (as pandas dataframes) 
X = spambase.data.features 
y = spambase.data.targets 
  
# metadata 
print(spambase.metadata) 
  
# variable information 
print(spambase.variables) 


{'uci_id': 94, 'name': 'Spambase', 'repository_url': 'https://archive.ics.uci.edu/dataset/94/spambase', 'data_url': 'https://archive.ics.uci.edu/static/public/94/data.csv', 'abstract': 'Classifying Email as Spam or Non-Spam', 'area': 'Computer Science', 'tasks': ['Classification'], 'characteristics': ['Multivariate'], 'num_instances': 4601, 'num_features': 57, 'feature_types': ['Integer', 'Real'], 'demographics': [], 'target_col': ['Class'], 'index_col': None, 'has_missing_values': 'no', 'missing_values_symbol': None, 'year_of_dataset_creation': 1999, 'last_updated': 'Mon Aug 28 2023', 'dataset_doi': '10.24432/C53G6X', 'creators': ['Mark Hopkins', 'Erik Reeber', 'George Forman', 'Jaap Suermondt'], 'intro_paper': None, 'additional_info': {'summary': 'The "spam" concept is diverse: advertisements for products/web sites, make money fast schemes, chain letters, pornography...\n\nThe classification task for this dataset is to determine whether a given email is spam or not.\n\t\nOur collecti

In [83]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.linear_model import LogisticRegression
import numpy as np



1. Use your implementation of Gradient Descent from Homework 2 and adapt it for logistic regression. Take 3 values of the learning rate and report the cross-entropy loss objective after 10, 50, and 100 iterations. At 100 iterations, report the accuracy, precision, recall, and F1 score for the 3 learning rates, and compare with the metrics given by the package on the training and testing sets.

In [87]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

X_train = np.hstack([np.ones((X_train.shape[0],1)), X_train.values if hasattr(X_train,'values') else X_train])
X_test  = np.hstack([np.ones((X_test.shape[0],1)),  X_test.values  if hasattr(X_test,'values') else X_test])

y_train = y_train.values.ravel() 
y_test  = y_test.values.ravel()  

def sigmoid(z):
    z = np.clip(z, -500, 500)
    return 1 / (1 + np.exp(-z))

# cross-entropy
def cross_entropy_loss(X, y, theta):
    p = sigmoid(X @ theta)
    p = np.clip(p, 1e-15, 1-1e-15)

    return - np.sum(y * np.log(p) + (1 - y) * np.log(1 - p))

def logistic_gd(X, y, alpha, num_iters):
    theta = np.zeros(X.shape[1])
    losses = {}
    
    for i in range(num_iters):
        p = sigmoid(X @ theta)
        p = np.clip(p, 1e-15, 1-1e-15)

        grad = X.T @ (p - y)
        theta = theta - alpha * grad

        if i+1 in [10, 50, 100]:
            losses[i+1] = cross_entropy_loss(X, y, theta)
    return theta, losses
    

results = {}
# 3 learning rates
for alpha in [0.001,0.01,0.1]:
    theta, losses = logistic_gd(X_train, y_train, alpha, 100)
    results[alpha] = theta
    print(f"\nLearning Rate = {alpha}")
    print("Cross-Entropy Loss:", losses)
    
    probs = sigmoid(X_test @ theta)
    preds = (probs >= 0.5).astype(int)


    accuracy = accuracy_score(y_test, preds)
    precision = precision_score(y_test, preds)
    recall = recall_score(y_test, preds)
    f1 = f1_score(y_test, preds)
    
    print("Accuracy:", accuracy)
    print("Precision:", precision)
    print("Recall:", recall)
    print("F1 Score:", f1)

print("\n--- Sklearn Logistic Regression ---")
logreg = LogisticRegression(max_iter=5000)
logreg.fit(X_train[:,1:], y_train)  

train_preds = logreg.predict(X_train[:,1:])
test_preds = logreg.predict(X_test[:,1:])

print("\nPackage Logistic Regression")

print("\nTraining Metrics")
print("Accuracy:", accuracy_score(y_train, train_preds))
print("Precision:", precision_score(y_train, train_preds))
print("Recall:", recall_score(y_train, train_preds))
print("F1:", f1_score(y_train, train_preds))

print("\nTesting Metrics")
print("Accuracy:", accuracy_score(y_test, test_preds))
print("Precision:", precision_score(y_test, test_preds))
print("Recall:", recall_score(y_test, test_preds))
print("F1:", f1_score(y_test, test_preds))



Learning Rate = 0.001
Cross-Entropy Loss: {10: 72947.58449582392, 50: 48289.23230289883, 100: 61810.865200290034}
Accuracy: 0.5091225021720244
Precision: 0.4552683896620278
Recall: 0.9642105263157895
F1 Score: 0.6185010128291695

Learning Rate = 0.01
Cross-Entropy Loss: {10: 72947.58449582392, 50: 48838.34353007993, 100: 70633.43210473965}
Accuracy: 0.43440486533449174
Precision: 0.4218472468916519
Recall: 1.0
F1 Score: 0.5933791380387258

Learning Rate = 0.1
Cross-Entropy Loss: {10: 72947.58449582392, 50: 48838.34316395389, 100: 70633.43130514224}
Accuracy: 0.43527367506516074
Precision: 0.4222222222222222
Recall: 1.0
F1 Score: 0.59375

--- Sklearn Logistic Regression ---

Package Logistic Regression

Training Metrics
Accuracy: 0.9289855072463769
Precision: 0.9266198282591726
Recall: 0.8871449925261584
F1: 0.9064528445971745

Testing Metrics
Accuracy: 0.9304952215464813
Precision: 0.9438202247191011
Recall: 0.8842105263157894
F1: 0.9130434782608695
